# Full 3,000-Question Diagnostic on Cloud GPU

Runs `src/run_full_diagnostic_3000q.py` against LLaVA-1.5-7B for one
POPE split. Instruments per-question CLIP-L max-sim, image attention
(layers 14-20), and absolute first-token logit gap. The output JSON
feeds `analysis/diagnostic_stats.py` (95 % bootstrap CIs, Cohen's d,
Mann-Whitney U).

Per-split wall-clock on a T4 in 4-bit: ~50-70 minutes. Run one split
per notebook session and combine the three JSONs at the end.

In [ ]:
SPLIT = 'adversarial'           # adversarial | popular | random
SAMPLES = 3000
MODEL_PATH = 'llava-hf/llava-1.5-7b-hf'
REPO_URL = 'https://github.com/Kesav2k04/ugaa-research.git'
REPO_BRANCH = 'ugaa-v2'

import os, sys, pathlib
IS_KAGGLE = os.path.exists('/kaggle/working')
IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
WORK = pathlib.Path('/kaggle/working') if IS_KAGGLE else (pathlib.Path('/content/work') if IS_COLAB else pathlib.Path.cwd() / 'cloud_run')
WORK.mkdir(parents=True, exist_ok=True)
print(f'kaggle={IS_KAGGLE} colab={IS_COLAB} work_dir={WORK}')

In [ ]:
import subprocess, sys
def pip(*a):
    print('pip', *a)
    return subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *a])
pip('transformers==4.40.1')
pip('accelerate>=0.28,<0.40', 'bitsandbytes>=0.43,<0.45')
pip('sentencepiece>=0.2.0', 'protobuf>=3.20,<5.0', 'safetensors')
pip('pandas', 'pyarrow', 'requests', 'Pillow', 'scipy', 'numpy<2.0')

In [ ]:
import subprocess, os
REPO_DIR = WORK / 'ugaa-research'
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--depth=1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--rebase'])
os.chdir(REPO_DIR)
print('cwd =', os.getcwd())

In [ ]:
import subprocess
subprocess.check_call([sys.executable, 'scripts/download_pope_full.py', '--splits', SPLIT])

In [ ]:
import os, subprocess
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/hf_cache/transformers'
os.makedirs('/tmp/hf_cache', exist_ok=True)

cmd = [
    sys.executable, 'src/run_full_diagnostic_3000q.py',
    '--split', SPLIT,
    '--samples', str(SAMPLES),
    '--model-path', MODEL_PATH,
    '--data-dir', 'datasets/pope',
    '--output-dir', 'experiments',
    '--device', 'cuda',
    '--cache-dir', '/tmp/hf_cache',
]
print(' '.join(cmd))
subprocess.check_call(cmd)

In [ ]:
import subprocess, sys
OUT = f'experiments/ugaa_full_{SPLIT}_{SAMPLES}q_diagnostic.json'
subprocess.check_call([sys.executable, 'analysis/diagnostic_stats.py', '--diagnostic', OUT, '--n-boot', '10000'])

In [ ]:
import shutil, pathlib
DEST = pathlib.Path('/kaggle/working/outputs') if IS_KAGGLE else (WORK / 'outputs')
DEST.mkdir(parents=True, exist_ok=True)
for p in pathlib.Path('experiments').glob('ugaa_full_*.json'):
    shutil.copy2(p, DEST / p.name)
    print('copied', p.name, p.stat().st_size, 'bytes')